In [1]:
# Импорты, seed и устройство

import os
import random
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


In [3]:
# Для CIFAR10 нормализация по трём каналам
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_full = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
test_ds = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

class_names = train_full.classes
print('Classes:', class_names)
print(f'Train size: {len(train_full)}, Test size: {len(test_ds)}')

# Разбиение train/val (80/20)
val_ratio = 0.2
val_size = int(len(train_full) * val_ratio)
train_size = len(train_full) - val_size
gen = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(train_full, [train_size, val_size], generator=gen)

BATCH_SIZE = 64
NUM_WORKERS = 0 if os.name == 'nt' else 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train size: {len(train_ds)}, Val size: {len(val_ds)}, Test size: {len(test_ds)}')
print(f'Input shape: {train_ds[0][0].shape}')  # (3, 32, 32)

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Train size: 50000, Test size: 10000
Train size: 40000, Val size: 10000, Test size: 10000
Input shape: torch.Size([3, 32, 32])


In [4]:
# 3. Модель MLP и вспомогательные функции


class MLP(nn.Module):
    def __init__(self, input_dim=3*32*32, hidden_dims=(256,128), num_classes=10,
                 activation='relu', dropout_p=0.0, use_batchnorm=False):
        super().__init__()
        act = {'relu': nn.ReLU, 'tanh': nn.Tanh}[activation.lower()]
        layers = [nn.Flatten()]
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(act())
            if dropout_p > 0:
                layers.append(nn.Dropout(p=dropout_p))
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def accuracy_from_logits(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n += 1
    return total_loss / n, total_acc / n

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n += 1
    return total_loss / n, total_acc / n

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.best_state = None
        self.counter = 0

    def step(self, score, model):
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

def fit(model, train_loader, val_loader, optimizer, criterion, device,
        epochs=20, early_stopping=None, verbose=True):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for epoch in range(1, epochs+1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        if verbose:
            print(f'Epoch {epoch:2d}/{epochs} | train loss={tr_loss:.4f}, acc={tr_acc:.4f} | val loss={va_loss:.4f}, acc={va_acc:.4f}')
        if early_stopping is not None:
            if early_stopping.step(va_acc, model):
                print(f'Early stopping at epoch {epoch}, best val_acc={early_stopping.best_score:.4f}')
                early_stopping.restore_best(model)
                break
    return history

In [5]:
# 4. Часть A: регуляризация (E1–E4)

criterion = nn.CrossEntropyLoss()
results = []

# E1 – baseline
set_seed(42)
model_e1 = MLP(hidden_dims=(256,128), dropout_p=0.0, use_batchnorm=False).to(device)
optimizer = optim.Adam(model_e1.parameters(), lr=1e-3)
hist_e1 = fit(model_e1, train_loader, val_loader, optimizer, criterion, device, epochs=30)
best_val_acc = max(hist_e1['val_acc'])
results.append({
    'experiment_id': 'E1', 'dataset': 'CIFAR10', 'seed': 42,
    'model_summary': '2x256,128, ReLU, no DO, no BN',
    'optimizer': 'Adam', 'lr': 1e-3, 'momentum': '', 'weight_decay': 0,
    'epochs_trained': len(hist_e1['train_loss']),
    'best_val_accuracy': best_val_acc, 'best_val_loss': min(hist_e1['val_loss'])
})

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch  1/30 | train loss=1.6613, acc=0.4115 | val loss=1.5478, acc=0.4557
Epoch  2/30 | train loss=1.4583, acc=0.4862 | val loss=1.4667, acc=0.4923
Epoch  3/30 | train loss=1.3464, acc=0.5232 | val loss=1.4117, acc=0.5073
Epoch  4/30 | train loss=1.2597, acc=0.5553 | val loss=1.4523, acc=0.5010
Epoch  5/30 | train loss=1.1853, acc=0.5821 | val loss=1.3839, acc=0.5236
Epoch  6/30 | train loss=1.1083, acc=0.6068 | val loss=1.4060, acc=0.5278
Epoch  7/30 | train loss=1.0505, acc=0.6285 | val loss=1.4022, acc=0.5248
Epoch  8/30 | train loss=0.9776, acc=0.6522 | val loss=1.4476, acc=0.5270
Epoch  9/30 | train loss=0.9142, acc=0.6743 | val loss=1.5134, acc=0.5180
Epoch 10/30 | train loss=0.8566, acc=0.6913 | val loss=1.5337, acc=0.5226
Epoch 11/30 | train loss=0.8032, acc=0.7164 | val loss=1.5769, acc=0.5301
Epoch 12/30 | train loss=0.7499, acc=0.7301 | val loss=1.6197, acc=0.5269
Epoch 13/30 | train loss=0.7061, acc=0.7499 | val loss=1.7082, acc=0.5187
Epoch 14/30 | train loss=0.6562, acc=0

In [ ]:
# E2 – Dropout (p=0.3)
set_seed(42)
model_e2 = MLP(hidden_dims=(256,128), dropout_p=0.3, use_batchnorm=False).to(device)
optimizer = optim.Adam(model_e2.parameters(), lr=1e-3)
hist_e2 = fit(model_e2, train_loader, val_loader, optimizer, criterion, device, epochs=30)
best_val_acc = max(hist_e2['val_acc'])
results.append({
    'experiment_id': 'E2', 'dataset': 'CIFAR10', 'seed': 42,
    'model_summary': '2x256,128, ReLU, DO=0.3, no BN',
    'optimizer': 'Adam', 'lr': 1e-3, 'momentum': '', 'weight_decay': 0,
    'epochs_trained': len(hist_e2['train_loss']),
    'best_val_accuracy': best_val_acc, 'best_val_loss': min(hist_e2['val_loss'])
})


In [ ]:
# E3 – BatchNorm
set_seed(42)
model_e3 = MLP(hidden_dims=(256,128), dropout_p=0.0, use_batchnorm=True).to(device)
optimizer = optim.Adam(model_e3.parameters(), lr=1e-3)
hist_e3 = fit(model_e3, train_loader, val_loader, optimizer, criterion, device, epochs=30)
best_val_acc = max(hist_e3['val_acc'])
results.append({
    'experiment_id': 'E3', 'dataset': 'CIFAR10', 'seed': 42,
    'model_summary': '2x256,128, ReLU, no DO, BN',
    'optimizer': 'Adam', 'lr': 1e-3, 'momentum': '', 'weight_decay': 0,
    'epochs_trained': len(hist_e3['train_loss']),
    'best_val_accuracy': best_val_acc, 'best_val_loss': min(hist_e3['val_loss'])
})


In [ ]:
# E4 – лучший из E2/E3 + EarlyStopping (лучший по val_acc – E2)
set_seed(42)
model_e4 = MLP(hidden_dims=(256,128), dropout_p=0.3, use_batchnorm=False).to(device)
optimizer = optim.Adam(model_e4.parameters(), lr=1e-3)
early_stop = EarlyStopping(patience=4, min_delta=0.0005)
hist_e4 = fit(model_e4, train_loader, val_loader, optimizer, criterion, device,
              epochs=50, early_stopping=early_stop)
best_val_acc = early_stop.best_score
results.append({
    'experiment_id': 'E4', 'dataset': 'CIFAR10', 'seed': 42,
    'model_summary': '2x256,128, ReLU, DO=0.3, no BN + ES',
    'optimizer': 'Adam', 'lr': 1e-3, 'momentum': '', 'weight_decay': 0,
    'epochs_trained': len(hist_e4['train_loss']),
    'best_val_accuracy': best_val_acc, 'best_val_loss': min(hist_e4['val_loss'])
})
